# 1- Importation des bibliothèques

In [18]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import psycopg2
import os
from dotenv import load_dotenv

# Charger les variables d'environnement
load_dotenv()

# Créer le dossier results
os.makedirs('../results', exist_ok=True)

print("✅ Toutes les bibliothèques sont importées")
print(f"📁 Dossier 'results' prêt")

✅ Toutes les bibliothèques sont importées
📁 Dossier 'results' prêt


# 2- Connexion à PostgreSQL

In [19]:
# Établir la connexion
conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
    database=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)

cursor = conn.cursor()

print("✅ Connexion à PostgreSQL établie")
print(f"📊 Base: {os.getenv('DB_NAME')}")

✅ Connexion à PostgreSQL établie
📊 Base: aircraft_db


# 3- Ajout de la colonne Année si elle n'existe pas

In [20]:
print("=" * 60)
print("🔧 AJOUT DE LA COLONNE ANNEE")
print("=" * 60)

# Vérifier si la colonne existe
cursor.execute("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_name = 'flight_summary_data' AND column_name = 'Annee'
""")

if cursor.fetchone() is None:
    cursor.execute('ALTER TABLE flight_summary_data ADD COLUMN "Annee" INTEGER')
    cursor.execute("""
        UPDATE flight_summary_data 
        SET "Annee" = CAST(SUBSTR("Date", 7, 4) AS INTEGER)
    """)
    conn.commit()
    print("✅ Colonne 'Annee' ajoutée")
else:
    print("✅ Colonne 'Annee' existe déjà")

cursor.execute('SELECT DISTINCT "Annee" FROM flight_summary_data ORDER BY "Annee"')
years = [str(y[0]) for y in cursor.fetchall()]
print(f"📅 Années: {', '.join(years)}")

🔧 AJOUT DE LA COLONNE ANNEE
✅ Colonne 'Annee' existe déjà
📅 Années: 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017


# 4- Chargement des données nettoyées dans un DataFrame

In [21]:
print("=" * 60)
print("📊 CHARGEMENT DES DONNÉES")
print("=" * 60)

query = """
SELECT 
    "Annee",
    "ASM_Domestic",
    COALESCE("ASM_International", 0) AS "ASM_International",
    "Flights_Domestic",
    COALESCE("Flights_International", 0) AS "Flights_International",
    "Passengers_Domestic",
    COALESCE("Passengers_International", 0) AS "Passengers_International",
    "RPM_Domestic",
    COALESCE("RPM_International", 0) AS "RPM_International",
    "Airline_Code",
    "Airport_Code"
FROM flight_summary_data
"""

df = pd.read_sql_query(query, conn)

# Création RPM total
df['RPM_Total'] = df['RPM_Domestic'] + df['RPM_International']

print(f"\n📊 Shape: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df.head())

📊 CHARGEMENT DES DONNÉES

📊 Shape: 1566 lignes, 12 colonnes
   Annee  ASM_Domestic  ASM_International  Flights_Domestic  \
0   2002         59854                0.0               774   
1   2002         55009                0.0               733   
2   2002         56586                0.0               745   
3   2003         57448                0.0               754   
4   2003         54006                0.0               674   

   Flights_International  Passengers_Domestic  Passengers_International  \
0                    0.0                60464                      59.0   
1                    0.0                57649                       0.0   
2                    0.0                66240                       0.0   
3                    0.0                55317                       4.0   
4                    0.0                53216                      65.0   

   RPM_Domestic  RPM_International Airline_Code Airport_Code  RPM_Total  
0         38363                0.0  

C:\Users\M.SH\AppData\Local\Temp\ipykernel_15820\3419003546.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


# 5- Correction des formats numériques

In [22]:
"""
Correction des formats numériques pour l'export CSV
"""

print("=" * 60)
print("🔧 CORRECTION DES FORMATS NUMÉRIQUES")
print("=" * 60)

# Liste des colonnes à traiter
colonnes_entiers = [
    'RPM_Domestic',
    'RPM_International', 
    'RPM_Total',
    'Flights_Domestic',
    'Flights_International',
    'Passengers_Domestic',
    'Passengers_International'
]

for col in colonnes_entiers:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).round(0).astype(int)
        print(f"   ✅ {col}: converti en entier (int)")

colonnes_float = [
    'ASM_Domestic',
    'ASM_International'
]

for col in colonnes_float:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).round(2)
        print(f"   ✅ {col}: converti en float avec 2 décimales")

print("\n✅ Correction terminée")

🔧 CORRECTION DES FORMATS NUMÉRIQUES
   ✅ RPM_Domestic: converti en entier (int)
   ✅ RPM_International: converti en entier (int)
   ✅ RPM_Total: converti en entier (int)
   ✅ Flights_Domestic: converti en entier (int)
   ✅ Flights_International: converti en entier (int)
   ✅ Passengers_Domestic: converti en entier (int)
   ✅ Passengers_International: converti en entier (int)
   ✅ ASM_Domestic: converti en float avec 2 décimales
   ✅ ASM_International: converti en float avec 2 décimales

✅ Correction terminée


# 6- Export des données nettoyées en CSV

In [23]:
print("=" * 60)
print("📁 EXPORT DES DONNÉES NETTOYÉES")
print("=" * 60)

df.to_csv('../results/flight_summary_clean.csv', index=False)
print(f"✅ flight_summary_clean.csv: {len(df)} lignes")

tables = ['aircraft', 'airlines', 'airports', 'individual_flights']
for table in tables:
    df_temp = pd.read_sql_query(f'SELECT * FROM {table}', conn)
    df_temp.to_csv(f'../results/{table}_clean.csv', index=False)
    print(f"✅ {table}_clean.csv: {len(df_temp)} lignes")

print("\n📁 Tous les CSV sont dans le dossier 'results'")

📁 EXPORT DES DONNÉES NETTOYÉES
✅ flight_summary_clean.csv: 1566 lignes


✅ aircraft_clean.csv: 5 lignes
✅ airlines_clean.csv: 3 lignes
✅ airports_clean.csv: 4 lignes
✅ individual_flights_clean.csv: 2270 lignes

📁 Tous les CSV sont dans le dossier 'results'


C:\Users\M.SH\AppData\Local\Temp\ipykernel_15820\1487338604.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_temp = pd.read_sql_query(f'SELECT * FROM {table}', conn)


# 7- Question 1 - Avion avec le plus de vols

In [24]:
"""
QUESTION 1: Quel avion a volé le plus ?
"""

print("=" * 70)
print("✈️ QUESTION 1: AVION AYANT EFFECTUÉ LE PLUS DE VOLS")
print("=" * 70)

query_q1 = """
SELECT 
    a."Aircraft_Type" AS avion,
    COUNT(f."Flight_Id") AS nombre_vols
FROM individual_flights f
INNER JOIN aircraft a ON f."Aircraft_Id" = a."Aircraft_Id"
GROUP BY a."Aircraft_Type"
ORDER BY nombre_vols DESC;
"""

df_q1 = pd.read_sql_query(query_q1, conn)

# 🔧 Correction: Renommer les colonnes pour Streamlit
df_q1.columns = ['avion', 'nombre_vols']

print("\n📊 Classement:")
for i, row in df_q1.iterrows():
    print(f"   {i+1}. {row['avion']}: {row['nombre_vols']:,} vols")

total = df_q1['nombre_vols'].sum()
print(f"\n🏆 GAGNANT: {df_q1.iloc[0]['avion']} avec {df_q1.iloc[0]['nombre_vols']:,} vols ({df_q1.iloc[0]['nombre_vols']/total*100:.1f}%)")

# Export
df_q1.to_csv('../results/question1_avions.csv', index=False)
print("\n💾 Exporté: question1_avions.csv")

✈️ QUESTION 1: AVION AYANT EFFECTUÉ LE PLUS DE VOLS

📊 Classement:
   1. Goose: 1,008 vols
   2. Thundercat: 553 vols
   3. Miniflock: 277 vols
   4. Bezantium: 240 vols
   5. Flockinator: 192 vols

🏆 GAGNANT: Goose avec 1,008 vols (44.4%)

💾 Exporté: question1_avions.csv


C:\Users\M.SH\AppData\Local\Temp\ipykernel_15820\1256634653.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q1 = pd.read_sql_query(query_q1, conn)


# 8- Graphique: Avion avec le plus de vols

In [25]:
fig = px.bar(
    df_q1,
    x='nombre_vols',
    y='avion',
    orientation='h',
    title='✈️ Nombre de vols par avion',
    labels={'nombre_vols': 'Nombre de vols', 'avion': 'Avion'},
    color='nombre_vols',
    color_continuous_scale='Blues',
    text='nombre_vols'
)

fig.update_traces(textposition='outside')
fig.update_layout(height=450)

fig.show()
fig.write_html('../results/fig_q1.html')
print("✅ Graphique sauvegardé: fig_q1.html")

✅ Graphique sauvegardé: fig_q1.html


# 9- Question 2 - Aéroport avec le plus de passagers

In [26]:
"""
QUESTION 2: Quel aéroport a transporté le plus de passagers ?
"""

print("=" * 70)
print("🛬 QUESTION 2: AÉROPORT AVEC LE PLUS DE PASSAGERS")
print("=" * 70)

query_q2 = """
WITH vols AS (
    SELECT "Departure_Airport_Code" AS code FROM individual_flights
    UNION ALL
    SELECT "Destination_Airport_Code" AS code FROM individual_flights
)
SELECT 
    v.code,
    COALESCE(a."Airport_Name", 'Inconnu') AS aeroport,
    COUNT(*) * 500 AS passagers
FROM vols v
LEFT JOIN airports a ON v.code = a."Airport_Code"
GROUP BY v.code, a."Airport_Name"
ORDER BY passagers DESC;
"""

df_q2 = pd.read_sql_query(query_q2, conn)

# 🔧 Correction: Renommer les colonnes pour Streamlit
df_q2.columns = ['code', 'aeroport', 'passagers']

print("\n📊 Classement:")
for i, row in df_q2.iterrows():
    print(f"   {i+1}. {row['aeroport']} ({row['code']}): {row['passagers']:,} passagers")

print(f"\n🏆 GAGNANT: {df_q2.iloc[0]['aeroport']} ({df_q2.iloc[0]['code']})")

# Export
df_q2.to_csv('../results/question2_aeroports.csv', index=False)
print("\n💾 Exporté: question2_aeroports.csv")

🛬 QUESTION 2: AÉROPORT AVEC LE PLUS DE PASSAGERS

📊 Classement:
   1. Flocktopia (FKT): 846,000 passagers
   2. Nestland Airport (NSA): 737,000 passagers
   3. Amazon Mothership (AMP): 680,500 passagers
   4. Aeroporte Inconnu (AA) (AA): 6,500 passagers

🏆 GAGNANT: Flocktopia (FKT)

💾 Exporté: question2_aeroports.csv


C:\Users\M.SH\AppData\Local\Temp\ipykernel_15820\304917153.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q2 = pd.read_sql_query(query_q2, conn)


# 10- Graphique : Aéroport avec le plus de passagers

In [27]:
fig = px.bar(
    df_q2,
    x='passagers',
    y='aeroport',
    orientation='h',
    title='🛬 Passagers par aéroport',
    labels={'passagers': 'Nombre de passagers', 'aeroport': 'Aéroport'},
    color='passagers',
    color_continuous_scale='Blues',
    text='passagers'
)

fig.update_traces(textposition='outside')
fig.update_layout(height=450)

fig.show()
fig.write_html('../results/fig_q2.html')
print("✅ Graphique sauvegardé: fig_q2.html")

✅ Graphique sauvegardé: fig_q2.html


# 11- Question 3 - Meilleure année pour le RPM

In [28]:
"""
QUESTION 3: Meilleure année pour le RPM par compagnie
"""

print("=" * 70)
print("📈 QUESTION 3: MEILLEURE ANNÉE POUR LE RPM")
print("=" * 70)

# RPM par année
df_rpm_yearly = df.groupby(['Airline_Code', 'Annee'])['RPM_Total'].sum().reset_index()
df_rpm_yearly.columns = ['compagnie', 'Annee', 'rpm_total']

# Meilleure année par compagnie
df_best = df_rpm_yearly.loc[df_rpm_yearly.groupby('compagnie')['rpm_total'].idxmax()]
df_best = df_best.rename(columns={'Annee': 'meilleure_annee'})

# 🔧 CORRECTION DES NOMBRES 
df_best['rpm_total'] = df_best['rpm_total'].round(0).astype(int)
df_rpm_yearly['rpm_total'] = df_rpm_yearly['rpm_total'].round(0).astype(int)

print("\n🏆 MEILLEURE ANNÉE PAR COMPAGNIE:")
for _, row in df_best.iterrows():
    print(f"   {row['compagnie']}: {int(row['meilleure_annee'])} (RPM_Total = {row['rpm_total']:,.0f})")

# Export
df_rpm_yearly.to_csv('../results/question3_rpm_yearly.csv', index=False)
df_best.to_csv('../results/question3_rpm_best_year.csv', index=False)
print("\n💾 Exportés: question3_rpm_yearly.csv, question3_rpm_best_year.csv")

📈 QUESTION 3: MEILLEURE ANNÉE POUR LE RPM

🏆 MEILLEURE ANNÉE PAR COMPAGNIE:
   AA: 2015 (RPM_Total = 11,912,594)
   FA: 2016 (RPM_Total = 17,318,668)
   GA: 2016 (RPM_Total = 49,260,222)

💾 Exportés: question3_rpm_yearly.csv, question3_rpm_best_year.csv


# 12- Graphique : Meilleure année pour le RPM

In [29]:
"""
Graphiques - Question 3
"""

# Graphique 1: Évolution
fig1 = px.line(
    df_rpm_yearly,
    x='Annee',
    y='rpm_total',
    color='compagnie',
    title='📈 Évolution du RPM Total',
    markers=True
)
fig1.update_layout(height=450)
fig1.show()
fig1.write_html('../results/fig_q3_evolution.html')

# Graphique 2: Meilleure année
fig2 = px.bar(
    df_best,
    x='compagnie',
    y='rpm_total',
    title='🏆 Meilleure année par compagnie',
    color='rpm_total',
    color_continuous_scale='Blues',
    text='meilleure_annee'
)
fig2.update_traces(textposition='outside', texttemplate='Année: %{text}')
fig2.update_layout(height=450)
fig2.show()
fig2.write_html('../results/fig_q3_best.html')

print("✅ Graphiques sauvegardés")

✅ Graphiques sauvegardés


# 13- Question 4 - Meilleure année pour la croissance

In [30]:
"""
QUESTION 4: Meilleure année pour la croissance (ASM)
"""

print("=" * 70)
print("📊 QUESTION 4: MEILLEURE ANNÉE POUR LA CROISSANCE")
print("=" * 70)

# ASM par année
df_asm = df.groupby(['Airline_Code', 'Annee'])['ASM_Domestic'].mean().reset_index()
df_asm.columns = ['compagnie', 'Annee', 'avg_asm']

# Meilleure année par compagnie
df_best_asm = df_asm.loc[df_asm.groupby('compagnie')['avg_asm'].idxmax()]
df_best_asm = df_best_asm.rename(columns={'Annee': 'meilleure_annee'})

# 🔧 CORRECTION DES NOMBRES 
df_best_asm['avg_asm'] = df_best_asm['avg_asm'].round(0).astype(int)
df_asm['avg_asm'] = df_asm['avg_asm'].round(2)

print("\n🏆 MEILLEURE ANNÉE PAR COMPAGNIE:")
for _, row in df_best_asm.iterrows():
    print(f"   {row['compagnie']}: {int(row['meilleure_annee'])} (AVG_ASM = {row['avg_asm']:,.0f})")

# Export
df_asm.to_csv('../results/question4_croissance_yearly.csv', index=False)
df_best_asm.to_csv('../results/question4_croissance_best_year.csv', index=False)
print("\n💾 Exportés: question4_croissance_yearly.csv, question4_croissance_best_year.csv")

📊 QUESTION 4: MEILLEURE ANNÉE POUR LA CROISSANCE

🏆 MEILLEURE ANNÉE PAR COMPAGNIE:
   AA: 2002 (AVG_ASM = 315,931)
   FA: 2016 (AVG_ASM = 427,255)
   GA: 2016 (AVG_ASM = 1,100,640)

💾 Exportés: question4_croissance_yearly.csv, question4_croissance_best_year.csv


# 14- Graphique : Meilleure année pour la croissance

In [31]:
"""
Graphiques - Question 4
"""

# Graphique 1: Évolution
fig1 = px.line(
    df_asm,
    x='Annee',
    y='avg_asm',
    color='compagnie',
    title='📊 Évolution de AVG(ASM)',
    markers=True
)
fig1.update_layout(height=450)
fig1.show()
fig1.write_html('../results/fig_q4_evolution.html')

# Graphique 2: Meilleure année
fig2 = px.bar(
    df_best_asm,
    x='compagnie',
    y='avg_asm',
    title='🏆 Meilleure année pour la croissance',
    color='avg_asm',
    color_continuous_scale='Blues',
    text='meilleure_annee'
)
fig2.update_traces(textposition='outside', texttemplate='Année: %{text}')
fig2.update_layout(height=450)
fig2.show()
fig2.write_html('../results/fig_q4_best.html')

print("✅ Graphiques sauvegardés")

✅ Graphiques sauvegardés


# 15- Fermeture de la connexion

In [32]:
cursor.close()
conn.close()

print("=" * 60)
print("✅ Connexion PostgreSQL fermée")
print("📁 Tous les résultats sont dans le dossier 'results'")
print("=" * 60)

# Liste des fichiers générés
print("\n📋 Fichiers générés:")
for f in os.listdir('../results'):
    print(f"   - {f}")

✅ Connexion PostgreSQL fermée
📁 Tous les résultats sont dans le dossier 'results'

📋 Fichiers générés:
   - aircraft_clean.csv
   - airlines_clean.csv
   - airports_clean.csv
   - fig_q1.html
   - fig_q1_avions.html
   - fig_q2.html
   - fig_q2_aeroports.html
   - fig_q3_best.html
   - fig_q3_best_year.html
   - fig_q3_dom_vs_int.html
   - fig_q3_evolution.html
   - fig_q4_asm_evolution.html
   - fig_q4_best.html
   - fig_q4_best_growth.html
   - fig_q4_evolution.html
   - flight_summary_clean.csv
   - individual_flights_clean.csv
   - question1_avions.csv
   - question2_aeroports.csv
   - question3_rpm_best.csv
   - question3_rpm_best_year.csv
   - question3_rpm_yearly.csv
   - question4_asm_best.csv
   - question4_asm_yearly.csv
   - question4_croissance_best_year.csv
   - question4_croissance_yearly.csv
